In [38]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader
from protgen.data import AA, FAMILIES, IGNORE_INDEX, special_tokens

In [39]:
train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")


# Inddexing
x, x_shifted, y, length = train_data[0]
print(f"# Sample 0: \nTokens: \n {x}\n Shifted: \n {x_shifted}\n Label: {y}\n Length: {length}")

# DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=True)

Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])
# Sample 0: 
Tokens: 
 tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])
 Shifted: 
 tensor([13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,  3,
        19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6, 22

In [40]:
print(f"vocab ({train_data.vocab_size} tokens): {train_data.vocab}")
print(f"families: {train_data.fam_to_idx}")
print(f"padding is ignored in the loss with ignore_index={IGNORE_INDEX}")

sequence = train_data.df["sequence"][0]
tokens = train_data.encode(sequence)
print()
print(f"encode / decode round-trips: {train_data.decode(tokens) == sequence}")
print(f"and from the shifted target too:  {train_data.decode(x_shifted) == sequence}")

vocab (23 tokens): {'<PAD>': 0, '<BOS>': 1, '<EOS>': 2, 'A': 3, 'C': 4, 'D': 5, 'E': 6, 'F': 7, 'G': 8, 'H': 9, 'I': 10, 'K': 11, 'L': 12, 'M': 13, 'N': 14, 'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'V': 20, 'W': 21, 'Y': 22}
families: {'AcyP': 0, 'HPr': 1, 'HU': 2}
padding is ignored in the loss with ignore_index=-1

encode / decode round-trips: True
and from the shifted target too:  True


In [41]:
x, x_shifted, y, length = next(iter(train_loader))
print(f"batch: x {tuple(x.shape)} {x.dtype}, x_shifted {tuple(x_shifted.shape)}, y {tuple(y.shape)}")

# Generative: one distribution over the vocabulary per position.
logits_gen = torch.randn(x.shape[0], train_data.vocab_size, train_data.max_length)
print(f"generative loss:     {F.cross_entropy(logits_gen, x_shifted, ignore_index=IGNORE_INDEX):.4f}")

# Classification: one distribution over the three families per sequence.
logits_fam = torch.randn(x.shape[0], len(FAMILIES))
print(f"classification loss: {F.cross_entropy(logits_fam, y):.4f}")

batch: x (32, 129) torch.int64, x_shifted (32, 129), y (32,)
generative loss:     3.5857
classification loss: 1.5602


In [42]:
# Build the AutoregressiveMLP
class AutoregressiveMLP(nn.Module):
    def __init__(self, vocab_size, context_size, embed_dim, hidden_dim):
        super().__init__()

        self.context_size = context_size
        self.net = nn.Sequential(
            nn.Embedding(vocab_size, embed_dim),
            nn.Flatten(),
            nn.Linear(context_size * embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
        )

    def forward(self, x):
        return self.net(x)

In [43]:
# Param setting
# Dataset parameters
vocab_size = train_data.vocab_size
pad_id = 0
bos_id = 1
eos_id = 2

# Model parameters
num_epochs = 100
context_size = 3 # 16, 32, 64, 128
embed_dim = 32
hidden_dim = 128
max_length = 128
# num_layers = 2
# dropout = 0.2

learning_rate = 1e-4

In [44]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [45]:
model = AutoregressiveMLP(
    vocab_size = vocab_size, 
    context_size = context_size, 
    embed_dim = embed_dim, 
    hidden_dim = hidden_dim
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [46]:
# MLP training
def train(model, train_loader, optimizer, device="cpu"):
    model.train()
    total_loss = 0

    for x, x_shifted, y, length in train_loader:
        x = x.to(device)
        x_shifted = x_shifted.to(device)
    
        padded_x = F.pad(x, (context_size-1, 0), value=pad_id)

        windows = padded_x.unfold(
            dimension = 1,
            size = context_size,
            step = 1
        )

        # B: batch_size; L: length; C: context_size
        B, L, C = windows.shape

        inputs = windows.reshape(B * L, C)
        targets = x_shifted.reshape(B * L)

        outputs = model(inputs)

        loss = F.cross_entropy(
            outputs,
            targets,
            ignore_index = IGNORE_INDEX
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    return avg_loss, 0

In [47]:
@torch.no_grad()
def validation(model, val_data, optimizer, device="cpu"):
    model.eval()
    val_loss = 0
    for x, x_shifted, y, length in val_data:
        x = x.to(device)
        x_shifted = x_shifted.to(device)
    
        padded_x = F.pad(x, (context_size-1, 0), value=pad_id)
        windows = padded_x.unfold(dimension=1, size=context_size, step=1)
        B, L, C = windows.shape
        inputs = windows.reshape(B * L, C)
        targets = x_shifted.reshape(B * L)
        outputs = model(inputs)

        val_loss += F.cross_entropy(outputs, targets, ignore_index=IGNORE_INDEX).item()
    
    avg_loss = val_loss / len(val_data)

    return avg_loss, 0

In [53]:
@torch.no_grad()
def generate(model, bos_id, eos_id, pad_id, device, max_length=128, temperature=1.0):
    model.eval()
    tokens = [bos_id]
    for _ in range(max_length):
        context = tokens[-model.context_size:]

        context = [pad_id] * (model.context_size - len(context)) + context

        x = torch.tensor([context], dtype=torch.long, device=device)

        logits = model(x)[0]

        logits[pad_id] = float("-inf")
        logits[bos_id] = float("-inf")

        probs = torch.softmax(logits / temperature, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1).item()

        if next_token == eos_id:
            break

        tokens.append(next_token)

    return tokens[1:]

In [49]:
best_val_loss = float("inf")

history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}
rows = []

patience = 10
counter = 0

for epoch in range(num_epochs):
    train_loss, train_acc = train(model, train_loader, optimizer, device)
    val_loss, val_acc = validation(model, val_loader, optimizer, device)
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    # history["train_acc"].append(train_acc)
    # history["val_acc"].append(val_acc)

    # find the best model
    if val_loss < best_val_loss:
        counter = 0

        best_val_loss = val_loss
        # best_val_acc = val_acc
        best_epoch = epoch
        torch.save(model, "best_MLP_model.pt")
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping")
        break
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")

print(f"The best model was saved at epoch {best_epoch+1} with val loss {best_val_loss:.4f}")

Epoch 1/100
Train Loss: 2.7746
Val Loss: 2.6063
Epoch 2/100
Train Loss: 2.5680
Val Loss: 2.5038
Epoch 3/100
Train Loss: 2.4855
Val Loss: 2.4539
Epoch 4/100
Train Loss: 2.4357
Val Loss: 2.4258
Epoch 5/100
Train Loss: 2.4036
Val Loss: 2.4097
Epoch 6/100
Train Loss: 2.3805
Val Loss: 2.4001
Epoch 7/100
Train Loss: 2.3631
Val Loss: 2.3912
Epoch 8/100
Train Loss: 2.3495
Val Loss: 2.3839
Epoch 9/100
Train Loss: 2.3383
Val Loss: 2.3785
Epoch 10/100
Train Loss: 2.3290
Val Loss: 2.3782
Epoch 11/100
Train Loss: 2.3210
Val Loss: 2.3703
Epoch 12/100
Train Loss: 2.3140
Val Loss: 2.3696
Epoch 13/100
Train Loss: 2.3077
Val Loss: 2.3677
Epoch 14/100
Train Loss: 2.3021
Val Loss: 2.3636
Epoch 15/100
Train Loss: 2.2970
Val Loss: 2.3633
Epoch 16/100
Train Loss: 2.2923
Val Loss: 2.3641
Epoch 17/100
Train Loss: 2.2880
Val Loss: 2.3625
Epoch 18/100
Train Loss: 2.2839
Val Loss: 2.3580
Epoch 19/100
Train Loss: 2.2803
Val Loss: 2.3562
Epoch 20/100
Train Loss: 2.2769
Val Loss: 2.3562
Epoch 21/100
Train Loss: 2.27

In [54]:
vocab = {v: k for k, v in train_data.vocab.items()}
model = torch.load("best_MLP_model.pt", 
                   map_location=device,
                   weights_only=False)
rows = []

for num in range(200):
    generated = generate(model, bos_id=bos_id, eos_id=eos_id, pad_id=pad_id, device=device, max_length=128)
    sequence = ''.join(
        vocab[i] for i in generated
    )
    rows.append({'sequence': sequence})
    
pd.DataFrame(rows).to_csv("generated_MLP_200.csv", index=False)

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt

# project = "/zhome/11/6/224005/DeepLearning/02456-autoregressive-project"

# gen = pd.read_csv(f"{project}/notebooks/generated.csv")
# train = pd.read_csv(f"{project}/data/train.csv")

# gen_len = gen["sequence"].str.len()
# train_len = train["sequence"].str.len()

# plt.figure(figsize=(7, 4))
# plt.hist(train_len, bins=40, density=True, alpha=0.5, label=f"train (n={len(train_len)})")
# plt.hist(gen_len, bins=20, density=True, alpha=0.5, label=f"MLP samples (n={len(gen_len)})")
# plt.xlabel("Sequence length (amino acids)")
# plt.ylabel("Density")
# plt.title("Length distribution: generated vs. training")
# plt.legend()
# plt.tight_layout()
# plt.savefig(f"{project}/notebooks/mlp_lengths.png", dpi=150)
# plt.show()

# print(gen_len.describe())